# 6장 예제 — 6.2 통계 기반 기법 `필수`

- **교재 출처**: manuscript/60_ch6_이상탐지.md
- **실행 방법**: example 폴더에서  python "6장/<이 파일>" (교재 코드의 data/ 상대경로를 그대로 쓰기 위해 작업 디렉터리는 example)
- **선행 필요**: 이 절의 코드는 앞 절에서 만든 변수를 이어 쓴다. 단독 실행이 필요하면 같은 장의 _장전체_실행본.py 를 쓰거나 아래 절을 먼저 실행하라 — 01_6-1_이상탐지_문제의_정의.py
- **실행 상태**: 단독 실행 확인됨(이 파일만 돌려도 끝까지 실행된다).

In [ ]:
# [환경 설정] data/ 경로 자동 감지
import os
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')

In [ ]:
# --- 실행 준비 (교재에서는 앞 절에서 이미 실행한 코드) --------------------
import numpy as np
import pandas as pd
df = pd.read_csv("data/battery_process_data.csv")
df["건조로_1구간_온도"] = df["건조로_1구간_온도"].replace(9999, np.nan)
df = df.fillna(df.median(numeric_only=True))   # 3.7절 clean_data() 규약

### z-score: 가장 단순한 출발점

In [ ]:
from scipy import stats

z = np.abs(stats.zscore(df["최종_용량_mAh"]))
out_z = df.loc[z > 3, "Lot_ID"].tolist()
print(f"용량 z-score>3 이상 Lot: {len(out_z)}개 → {out_z}")

# 교재 실행 결과 ------------------------------------------------------
#   용량 z-score>3 이상 Lot: 4개 → ['L25246', 'L25627', 'L25723', 'L25759']

### 수정 z-score: 중앙값과 MAD

In [ ]:
x = df["최종_용량_mAh"]
med = np.median(x)
mad = np.median(np.abs(x - med))
mz = 0.6745 * (x - med) / mad
out_mz = df.loc[np.abs(mz) > 3.5, "Lot_ID"].tolist()
print(f"수정 z-score>3.5 이상 Lot: {len(out_mz)}개 → {out_mz}")

# 교재 실행 결과 ------------------------------------------------------
#   수정 z-score>3.5 이상 Lot: 1개 → ['L25759']

### 변수를 늘리면 생기는 문제: 다중 비교

In [ ]:
feats = ["믹싱_RPM", "믹싱_온도", "코팅_토출압력", "건조로_1구간_온도",
         "프레스_압력", "프레스_Gap",              # 공정 변수 6개
         "최종_용량_mAh", "전극_면저항_mOhmcm2"]    # 품질 변수 2개

zall = np.abs(stats.zscore(df[feats]))
print("어느 한 변수라도 |z|>3인 Lot:", (zall > 3).any(axis=1).sum(), "개")

# 교재 실행 결과 ------------------------------------------------------
#   어느 한 변수라도 |z|>3인 Lot: 12 개

### 다변량으로: 마할라노비스 거리

In [ ]:
import textwrap   # 긴 한 줄 출력을 지정한 폭에서 접어 주는 파이썬 표준 모듈

# feats는 6.2.3절에서 정의한 8개 변수 목록을 그대로 사용한다
X = df[feats].values

mu = X.mean(axis=0)
Sigma_inv = np.linalg.inv(np.cov(X.T))
d2 = np.einsum("ij,jk,ik->i", X - mu, Sigma_inv, X - mu)  # D_M^2

thr = stats.chi2.ppf(0.99, df=len(feats))  # χ²(8)의 99% 분위수
out_maha = df.loc[d2 > thr, "Lot_ID"].tolist()
print(f"임계값 D²>{thr:.2f} → 이상 Lot {len(out_maha)}개")
# 그냥 print(out_maha)로도 되지만 한 줄이 너무 길어지므로 62자에서 접어 출력한다.
# subsequent_indent=" "는 두 번째 줄부터 한 칸 들여쓰라는 뜻이다(내용은 그대로).
print(textwrap.fill(str(out_maha), width=62, subsequent_indent=" "))

# 교재 실행 결과 ------------------------------------------------------
#   임계값 D²>20.09 → 이상 Lot 11개
#   ['L25053', 'L25141', 'L25166', 'L25281', 'L25408', 'L25616',
#    'L25627', 'L25723', 'L25829', 'L25913', 'L25949']